# Interactive hackingtool workspace

This notebook is ready for **Run All** in the repository's GitHub Codespace. Commands run as `root` inside the isolated container, not on the Codespaces host.

The notebook prepares a non-blocking launcher and a safe argument-based command helper. Run security tools only against systems you own or are explicitly authorized to test.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
from pathlib import Path
from typing import Sequence


def find_repo_root(start: Path | None = None) -> Path:
    """Find the src-layout repository without depending on notebook launch directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "hackingtool").is_dir():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml and src/hackingtool from the current directory")


REPO_ROOT = find_repo_root()
IS_ROOT = (os.geteuid() == 0) if hasattr(os, "geteuid") else False
IS_ISOLATED_CONTAINER = Path("/.dockerenv").exists() or bool(
    os.environ.get("CODESPACES") or os.environ.get("REMOTE_CONTAINERS")
)


def run_command(
    arguments: Sequence[str | os.PathLike[str]],
    *,
    cwd: Path = REPO_ROOT,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run an explicit argument list without shell interpolation."""
    if not arguments:
        raise ValueError("At least one command argument is required")
    command = [os.fspath(argument) for argument in arguments]
    print("Running:", subprocess.list2cmdline(command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)


def launch_hackingtool() -> subprocess.CompletedProcess[str]:
    """Launch the installed interactive CLI on demand; this is not called by Run All."""
    executable = shutil.which("hackingtool")
    if executable is None:
        raise RuntimeError("hackingtool CLI is not installed; rebuild the Codespace or run: python -m pip install -e .")
    return run_command([executable], check=False)


In [ ]:
# Run-All readiness check.
required_paths = [
    REPO_ROOT / "pyproject.toml",
    REPO_ROOT / "src" / "hackingtool" / "cli.py",
    REPO_ROOT / "src" / "hackingtool" / "constants.py",
]
missing_paths = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required project paths: {missing_paths}")

if importlib.util.find_spec("hackingtool") is None:
    raise RuntimeError("hackingtool package is not installed in this kernel; rebuild the Codespace or run: python -m pip install -e .")

if shutil.which("hackingtool") is None:
    raise RuntimeError("hackingtool console script is not on PATH")

write_probe = REPO_ROOT / ".interactive-notebook-write-check"
try:
    write_probe.write_text("ok", encoding="utf-8")
finally:
    write_probe.unlink(missing_ok=True)

print(f"Repository:          {REPO_ROOT}")
print(f"Python package:      {importlib.util.find_spec('hackingtool').origin}")
print(f"CLI:                 {shutil.which('hackingtool')}")
print(f"Root in container:   {IS_ROOT}")
print(f"Container detected:  {IS_ISOLATED_CONTAINER}")
print("\nRun All is complete. Call launch_hackingtool() when you want the interactive CLI.")
